# Generation of datasets to train MACE models
In this tutorial, we will generate a datasaet that we can then use to train a MACE model to simulate the SN2 equilibrium reaction in the gas phase: 

$CH_{3}Br + Cl^{-} \rightleftharpoons CH_{3}Cl + Br^{-}$

The dataset will be generated by running single point calculations on a set of structures, then we will extract the energy, forces, and stress tensor and add it to the original ```Atoms``` object. Finally, we will write the data set to an ```extxyz``` file.

For the purpose of this tutorial, we will use the xTB method, which is less accurate than PBE but much faster, to calculate energies and forces and stresses for a dataset of 100 random structures taken from two pools of snapshots from classical MD simulations of the $CH_{3}Br + Cl^{-}$ and $CH_{3}Cl + Br^{-}$ systems.



## Import relevant modules
Apart from the usual suspects, we need to import the `mk_mace_dataset` function from the `pycp2k.workflows` module. This will allow us to generate a list of ASE Atoms objects, each with the energy, forces, and stress tensor calculated.

In [ ]:
from ase.io import write
import subprocess
import random
import os
from pycp2k.templates.GLOBAL.GLOBAL import CP2K
from pycp2k.templates.FORCE_EVAL.xTB_templates import add_xTB_OT
from pycp2k.templates.PRINT.singlepoint import *
from pycp2k.workflows.mk_mace_dataset import load_dataset,get_elements,add_isolated_atoms

## Loading the dataset and adding the isolated atoms
We will use the `load_dataset` function to load the datasets from the `CH3Br_Cl-.xyz` and `CH3Cl_Br-.xyz` files. These files will be read as extxyz and all the structures in them have a specified charge of -1. This function will return a list of ASE Atoms objects, each of them including the charge and a boolean flag indicating if the structure has an odd number of electrons.

```python
from pycp2k.workflows.mk_mace_dataset import load_dataset
ds=load_dataset("CH3Br_Cl-.xyz")
ds=ds+load_dataset("CH3Cl_Br-.xyz")
```
Once both datasets have been loaded, we will shuffle them to randomize the order of the structures.

```python
random.shuffle(ds)
```

We will also use the `get_elements` function to get the set of elements in the dataset, then add them as isolated atoms.

```python
from pycp2k.workflows.mk_mace_dataset import get_elements,add_isolated_atoms
symbols_set=get_elements(ds)
ds=add_isolated_atoms(ds,symbols_set)
```

Finally, for the purpose of this tutorial, we will only use the first 100 structures in the dataset, plus the isolated atoms.

In [ ]:
ds=load_dataset("CH3Br_Cl-.xyz")
ds=ds+load_dataset("CH3Cl_Br-.xyz")
random.shuffle(ds)
symbols_set=get_elements(ds)
ds=add_isolated_atoms(ds,symbols_set)
ds_chunk=ds[0:100+len(symbols_set)]
for system in ds_chunk:
    print(system, system.info)

## Single point calculations and postprocessing
This is the main loop. For each structure in the data set (including the isolated atoms), we will run a single point calculation using CP2K, then we will extract energies and forces and add them to the original ```Atoms``` objects. Finally, we will write the data set to an ```extxyz``` file. In this example, we will use xTB with only 3 inner SCF steps and 1 outer SCF step, for the sake of speed.

First, we will define the calculator. Since the systems have all the same atoms, there is no reason why we couldn't use the same calculator and only change the project name and atom coordinates. However, if we were to run in parallel, or with systems with different atoms, we would need to define a new calculator for each system, so that's what we will do here every time:

```python
calc=CP2K(project_name="mace_xTB",
          run_type="ENERGY_FORCE",
          cp2k_command="/opt/homebrew/bin/cp2k.psmp")
```

Then, we will add the xTB calculator to the system:

```python
add_xTB_OT(atoms=system,calc=calc,
           charge=system.info.get("charge",0),
           LSD=system.info["oddNumberofElectrons"], # This flag is necessary when there is an odd number of electrons
           Ignore_convergence_failure=True, # Only for the sake of speed of this tutorial
           max_scf=3,outer_max_scf=1)
```

The energy comes from the standard CP2K output, but the forces are written to a separate file, so we need to add a print option for that (those functions are defined in the `pycp2k.templates.PRINT` module and were imported using the `from pycp2k.templates.PRINT.singlepoint import *` statement).

```python
forces_path=add_print_singlepoint_forces(calc=calc,filename="forces",unit=None)
```

Then, the try-except block is used to run the calculation and postprocess the output. If a calculation fails (it shouldn't), we will print the last 30 lines of the output file and continue with the next structure.

In [ ]:
failed_calculations=[]
successful_calculations_idx=[]
root_dir=os.getcwd()
for i in range(len(ds_chunk)):
    os.makedirs(f"{root_dir}/results/mace_xTB_{i}",exist_ok=True)
    os.chdir(f"{root_dir}/results/mace_xTB_{i}")
    print(os.getcwd())
    system=ds_chunk[i]
    calc=CP2K(project_name=f"mace_xTB_{i}",
              run_type="ENERGY_FORCE",
              cp2k_command="/opt/homebrew/bin/cp2k.psmp")
    add_xTB_OT(atoms=system,calc=calc,
               charge=system.info.get("charge",0),
               LSD=system.info["oddNumberofElectrons"],
               Ignore_convergence_failure=True,
               max_scf=3,outer_max_scf=1)
    forces_path=add_print_singlepoint_forces(calc=calc,filename="forces",unit="EV/ANGSTROM")
    stress_path=add_print_stress_tensor(calc=calc,filename="./",unit="EV/ANGSTROM^3")
    try:
        calc.run()
        system.info["E"]=postprocess_energy(calc=calc)
        system.set_array("forces",postprocess_forces(forces_path=forces_path))
        successful_calculations_idx.append(i)
    except Exception as e:
        print(f"Error: {e}")
        output_file=f"{calc.project_name}.out"
        subprocess.run(["tail", "-n", "30", output_file])
        failed_calculations.append(calc.project_name)
        continue
    finally:
        os.chdir(root_dir)

print("Failed calculations:")
for calc in failed_calculations:
    print(calc)

In [ ]:
ds_out=[]
for idx in successful_calculations_idx:
    ds_out.append(ds_chunk[idx])
write("ds_ready.xyz",ds_out,format="extxyz")
